## AND-105 Task 8: Evaluation and Production Recommendation

The explanations in `predictions.risk_explanation` were written in Task 7 by the
local model `llama3.1:8b-instruct-q8_0` (12,215 high-risk elevators). Here I check
whether they're actually any good — are the facts right, would they help a
technician, and are similar elevators described the same way — and then decide what
to use in production.

I took 10 high-risk elevators as my sample and pull the ground truth straight from
the database so I'm not grading against my own memory. Needs the `aztia-db-1`
container running.

In [15]:
import os, re, textwrap, json
from dataclasses import dataclass, field
from enum import Enum
from collections import defaultdict
import psycopg2, psycopg2.extras

DB = dict(host="localhost", port=5432, dbname="rocket_elevators",
          user="rocket_user", password="rocket_pass")

# What I'm grading. To grade a different model, write its output to another
# column and change EXPLANATION_COLUMN (or pass a dict to evaluate_model below).
MODEL_LABEL = "llama3.1:8b-instruct-q8_0"
EXPLANATION_COLUMN = "risk_explanation"
SAMPLE = [8, 9, 70, 74, 75, 76, 79, 80, 83, 99]

conn = psycopg2.connect(**DB)

### Scoring scale

I score each explanation as **correct** (all facts match), **minor** (wrong detail,
right idea — e.g. a year that isn't in the data), **major** (a claim the data
contradicts), or **hallucination** (a claim with nothing behind it).

In [16]:
class Score(Enum):
    CORRECT = "correct"
    MINOR = "minor"
    MAJOR = "major"
    HALLUCINATION = "hallucination"

# My own read of the sample after eyeballing each one. I only keep these to prove
# the automated checker below agrees with me — a new model doesn't need re-reading.
my_labels = {8:"CORRECT", 9:"CORRECT", 70:"MAJOR", 74:"CORRECT", 75:"CORRECT",
             76:"CORRECT", 79:"CORRECT", 80:"CORRECT", 83:"CORRECT", 99:"CORRECT"}

### The ground truth and the explanations

Everything I check against — license status, inspection counts, pass rate, the most
recent inspection, the years inspections happened — comes from these queries, so it
stays true no matter which model I'm grading.

In [17]:
def ground_truth(conn, ids):
    t = {}
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute("""
            SELECT e.elevator_id, e.license_status,
                   COUNT(i.*) AS total,
                   COUNT(*) FILTER (WHERE i.outcome='Passed') AS passed,
                   ROUND(COUNT(*) FILTER (WHERE i.outcome='Passed')::numeric
                         / NULLIF(COUNT(i.*),0)*100, 1) AS pass_rate
            FROM elevators e LEFT JOIN inspections i ON i.elevator_id = e.elevator_id
            WHERE e.elevator_id = ANY(%s)
            GROUP BY e.elevator_id, e.license_status""", (ids,))
        for r in cur.fetchall():
            t[r["elevator_id"]] = dict(r)
        cur.execute("""
            SELECT DISTINCT ON (elevator_id) elevator_id,
                   latest_inspection_date AS date, outcome
            FROM inspections WHERE elevator_id = ANY(%s)
            ORDER BY elevator_id, latest_inspection_date DESC""", (ids,))
        for r in cur.fetchall():
            t[r["elevator_id"]]["recent"] = f"{r['date']} ({r['outcome']})"
            t[r["elevator_id"]]["recent_year"] = str(r["date"])[:4]
        cur.execute("""
            SELECT elevator_id, array_agg(DISTINCT latest_inspection_date) AS dates
            FROM inspections WHERE elevator_id = ANY(%s) GROUP BY elevator_id""", (ids,))
        for r in cur.fetchall():
            t[r["elevator_id"]]["years"] = {str(d)[:4] for d in r["dates"]}
    return t

def load_explanations(conn, ids, column=EXPLANATION_COLUMN):
    with conn.cursor() as cur:
        cur.execute(f"SELECT elevator_id, {column} FROM predictions "
                    f"WHERE elevator_id = ANY(%s)", (ids,))
        return dict(cur.fetchall())

truth = ground_truth(conn, SAMPLE)
expl = load_explanations(conn, SAMPLE)

for e in SAMPLE:
    t = truth[e]
    print(f"{e:>3}  {t['license_status']:11} {t['total']} insp, {t['passed']} passed, "
          f"{t['pass_rate']}%  last: {t['recent']}")

  8  ACTIVE      6 insp, 1 passed, 16.7%  last: 2015-03-27 (Follow up)
  9  ACTIVE      7 insp, 1 passed, 14.3%  last: 2015-03-27 (DC Follow up)
 70  BY REQUEST  1 insp, 0 passed, 0.0%  last: 2011-12-28 (Follow up)
 74  TERMINATED  2 insp, 0 passed, 0.0%  last: 2014-04-08 (Vol Shut Down)
 75  TERMINATED  2 insp, 0 passed, 0.0%  last: 2014-04-10 (Vol Shut Down)
 76  TERMINATED  2 insp, 1 passed, 50.0%  last: 2014-04-08 (Vol Shut Down)
 79  TERMINATED  2 insp, 1 passed, 50.0%  last: 2014-04-10 (Vol Shut Down)
 80  TERMINATED  4 insp, 2 passed, 50.0%  last: 2014-04-10 (Vol Shut Down)
 83  TERMINATED  4 insp, 2 passed, 50.0%  last: 2014-04-08 (Vol Shut Down)
 99  BY REQUEST  1 insp, 0 passed, 0.0%  last: 2011-04-27 (Follow up)


### Checking the facts automatically

Rather than hand-grade every model I ever try, I pull the numbers out of the text
(pass rate, totals, passed count, any years it mentions) and compare them to the
database. The same function works on any model's output.

In [18]:
NUM = r"(\d+(?:\.\d+)?|zero|one|two|three|four|five|six|seven|eight|nine|ten)"
WORDS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,
         "six":6,"seven":7,"eight":8,"nine":9,"ten":10}

def to_int(tok):
    tok = tok.lower().strip()
    return int(float(tok)) if re.fullmatch(r"\d+(?:\.\d+)?", tok) else WORDS.get(tok)

def claims(text):
    t = text.lower()
    c = {}
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", t)
    if m: c["pass_rate"] = float(m.group(1))
    elif "zero pass rate" in t: c["pass_rate"] = 0.0
    m = (re.search(NUM + r"\s+total", t) or re.search(r"total of " + NUM, t)
         or re.search(NUM + r"\s+(?:total\s+)?inspections", t))
    if m: c["total"] = to_int(m.group(1))
    m = re.search(NUM + r"\s+out of\s+" + NUM, t)
    if m:
        c["passed"] = to_int(m.group(1)); c.setdefault("total", to_int(m.group(2)))
    else:
        m = (re.search(r"(?:passed|passing)\s+(?:only\s+)?" + NUM, t)
             or re.search(r"(?:only\s+)?" + NUM + r"\s+(?:inspections?\s+)?(?:passed|passing)", t))
        if m: c["passed"] = to_int(m.group(1))
    if "passed" not in c and any(p in t for p in
            ("none have passed", "none passed", "failing all", "perfect record of failing")):
        c["passed"] = 0
    c["years"] = set(re.findall(r"(?:19|20)\d{2}", t))
    return c

def grade(eid, text, t):
    c, issues, score = claims(text), [], Score.CORRECT
    if c.get("pass_rate") == 0.0 and c.get("passed", 0):
        issues.append(f"says {c['passed']} passed but also 0.0% pass rate"); score = Score.MAJOR
    if c.get("pass_rate") is not None and t["pass_rate"] is not None \
            and abs(c["pass_rate"] - float(t["pass_rate"])) > 0.15:
        issues.append(f"pass rate {c['pass_rate']}% vs real {t['pass_rate']}%"); score = Score.MAJOR
    if "total" in c and c["total"] != t["total"]:
        issues.append(f"total {c['total']} vs real {t['total']}"); score = Score.MAJOR
    if "passed" in c and c["passed"] != t["passed"]:
        issues.append(f"passed {c['passed']} vs real {t['passed']}"); score = Score.MAJOR
    for y in c["years"]:
        if t["years"] and y not in t["years"]:
            issues.append(f"mentions {y}, not an inspection year")
            if score is Score.CORRECT: score = Score.MINOR
    return score, issues

Quick sanity check that the automated grader matches what I read by hand:

In [19]:
ok = 0
for e in SAMPLE:
    score, _ = grade(e, expl[e], truth[e])
    match = score.name == my_labels[e]
    ok += match
    print(f"{e:>3}  auto={score.name:6} me={my_labels[e]:6} {'ok' if match else 'MISMATCH'}")
print(f"\n{ok}/{len(SAMPLE)} agree — good enough to trust the automated runs.")
assert ok == len(SAMPLE)

  8  auto=CORRECT me=CORRECT ok
  9  auto=CORRECT me=CORRECT ok
 70  auto=MAJOR  me=MAJOR  ok
 74  auto=CORRECT me=CORRECT ok
 75  auto=CORRECT me=CORRECT ok
 76  auto=CORRECT me=CORRECT ok
 79  auto=CORRECT me=CORRECT ok
 80  auto=CORRECT me=CORRECT ok
 83  auto=CORRECT me=CORRECT ok
 99  auto=CORRECT me=CORRECT ok

10/10 agree — good enough to trust the automated runs.


## Accuracy

Does each explanation actually line up with the data? I went through all ten, pulled
the numbers out of each one, and put them next to what the database says.

In [20]:
dist = defaultdict(int)
for e in SAMPLE:
    score, issues = grade(e, expl[e], truth[e])
    dist[score] += 1
    t = truth[e]
    print(f"========== elevator {e}  ->  {score.name} ==========")
    print(textwrap.fill(expl[e], 88))
    print(f"  checked against DB: {t['license_status']}, {t['total']} inspections, "
          f"{t['passed']} passed, {t['pass_rate']}% pass rate, last {t['recent']}")
    if issues:
        for i in issues: print("  ! ", i)
    else:
        print("  -> every number matches")
    print()

print("Tally:", {s.name: dist[s] for s in Score})
print(f"Clean: {dist[Score.CORRECT]}/{len(SAMPLE)}")

========== elevator 8  ->  CORRECT ==========
With a risk score of 0.9534, this elevator poses high risk. The inspection outcome
summary shows that out of 6 total inspections, only 1 passed, resulting in a pass rate
of 16.7%.
  checked against DB: ACTIVE, 6 inspections, 1 passed, 16.7% pass rate, last 2015-03-27 (Follow up)
  -> every number matches

========== elevator 9  ->  CORRECT ==========
With a risk score of 0.8910, this elevator is at high risk and requires imminent
compliance action. The inspection history shows a pass rate of 14.3% out of 7 total
inspections, with only 1 passing and 6 requiring action.
  checked against DB: ACTIVE, 7 inspections, 1 passed, 14.3% pass rate, last 2015-03-27 (DC Follow up)
  -> every number matches

========== elevator 70  ->  MAJOR ==========
With a risk score of 0.8165, this elevator poses high risk due to its history of failing
inspections, with a pass rate of 0.0% and only one inspection passing out of a total of
one. The most recent inspec

Nine of the ten hold up. The odd one out is elevator 70: it says one inspection passed
but in the same breath reports a 0% pass rate, and the database confirms 0 of 1 passed.
For a compliance tool that's the worst way to be wrong, since someone skimming it could
walk away thinking the device passed. On the upside, none of them invented a date.

## Usefulness

Would any of this actually help a tech on the floor? Three things I looked at: whether
it points at something concrete instead of just "this is risky", whether it says what to
go check, and whether it's short enough that someone would read it rather than skip past
it. For that last one I just count words as a rough stand-in.

In [21]:
# "inspect" matched on a word boundary so it doesn't fire on "inspection",
# which shows up in every single explanation.
ACTION = re.compile(r"\b(inspect|re-?inspect|check|schedule|prioriti[sz]e|"
                    r"recommend\w*|should|investigate|verify|examine|look for|"
                    r"focus on|ensure)\b")
SPECIFIC = ("pass rate", "%", "inspection", "terminated", "shut down",
            "follow up", "follow-up", "license")

spec = act = conc = 0
print(f"{'elev':>4} {'words':>5} {'specific':>9} {'actionable':>11} {'brief':>6}")
for e in SAMPLE:
    t = expl[e].lower(); w = len(expl[e].split())
    s = sum(h in t for h in SPECIFIC) >= 2 and bool(re.search(r"\d", t))
    a = bool(ACTION.search(t))
    c = w <= 75   # short enough to skim
    spec += s; act += a; conc += c
    print(f"{e:>4} {w:>5} {str(s):>9} {str(a):>11} {str(c):>6}")
print(f"\nspecific {spec}/10   actionable {act}/10   brief {conc}/10")

elev words  specific  actionable  brief
   8    32      True       False   True
   9    39      True       False   True
  70    48      True       False   True
  74    39      True       False   True
  75    35      True       False   True
  76    38      True       False   True
  79    48      True       False   True
  80    52      True       False   True
  83    44      True       False   True
  99    36      True       False   True

specific 10/10   actionable 0/10   brief 10/10


They do well on two of the three: every explanation names concrete things (the score,
pass rate, counts, license status) and they're all short. Where they fall flat is
telling you what to actually do — not one of them suggests something to go inspect. They
describe the risk and stop there. To be fair the Task 7 prompt never asked for a
recommendation, so this is a prompt fix more than a model failing.

## Consistency

Two elevators with the same numbers should read more or less the same way. I group the
sample by identical profile (same license status, counts, pass rate) and check whether
each explanation surfaces the same facts.

In [22]:
def mentions(text):
    t = text.lower()
    return {
        "license":  "terminat" in t or "license" in t,
        "shutdown": "shut down" in t or "shutdown" in t or "follow" in t,
        "date":     bool(re.search(r"\b(?:19|20)\d{2}\b", t)),
    }

groups = defaultdict(list)
for e in SAMPLE:
    t = truth[e]
    groups[(t["license_status"], t["total"], t["passed"], float(t["pass_rate"]))].append(e)

for ids in [g for g in groups.values() if len(g) >= 2]:
    a, b = ids[0], ids[1]
    ma, mb = mentions(expl[a]), mentions(expl[b])
    p = truth[a]
    print(f"{p['license_status']}, {p['total']} insp, {p['passed']} passed, "
          f"{p['pass_rate']}%  ->  {a} vs {b}")
    for f in ("license", "shutdown", "date"):
        flag = "same" if ma[f] == mb[f] else ">>> differs"
        print(f"    {f:9} {a}:{str(ma[f]):5} {b}:{str(mb[f]):5}  {flag}")
    print()

BY REQUEST, 1 insp, 0 passed, 0.0%  ->  70 vs 99
    license   70:False 99:False  same
    shutdown  70:True  99:False  >>> differs
    date      70:True  99:False  >>> differs

TERMINATED, 2 insp, 0 passed, 0.0%  ->  74 vs 75
    license   74:True  75:True   same
    shutdown  74:True  75:False  >>> differs
    date      74:True  75:False  >>> differs

TERMINATED, 2 insp, 1 passed, 50.0%  ->  76 vs 79
    license   76:False 79:True   >>> differs
    shutdown  76:True  79:True   same
    date      76:True  79:True   same

TERMINATED, 4 insp, 2 passed, 50.0%  ->  80 vs 83
    license   80:True  83:True   same
    shutdown  80:True  83:False  >>> differs
    date      80:True  83:False  >>> differs



More often than not, identical profiles come out described differently. 76 and 79 look
the same on paper but only 79 bothers to mention the terminated license; 80 and 83 match
but only 80 brings up the 2014 shutdown. So what gets highlighted is more or less luck of
the draw.

## What I found

Pulling the three dimensions together:

**Accuracy.** 9/10 correct, one major (elevator 70's self-contradiction), zero
hallucinations. The numbers are almost always copied straight from the data and the
dates check out.

**Where it slips.** The worst case is inventing a passed count that contradicts the pass
rate it just stated, which showed up on a single-inspection elevator. Beyond that, it's
inconsistent: elevators that look identical in the data don't always mention the same
things (license status, the most recent serious outcome).

**Usefulness.** It leads with the score every time and stays short, which is good. The
gap is that none of them say what to inspect — they stop at describing the risk.

## Production recommendation

I'd move to Claude **`claude-sonnet-4-6`** and generate the backfill through the
**Batches API**. The local 8B model gets it right about 90% of the time, but for
something that feeds compliance decisions the other 10% is the problem — a confidently
wrong pass count like elevator 70's is exactly the kind of thing you can't put in front
of an inspector.

Two reasons it's the right pick. First, the stronger instruction-following should clear
up the arithmetic contradictions. Second, I'd pin the output to a fixed set of fields
(score, license, pass rate, most recent outcome, recommended action) using structured
outputs — that makes the consistency problem disappear by construction, because every
elevator is forced through the same shape, and it gives me a clean place to add the
action line that's missing today.

Haiku 4.5 is the fallback if cost ever becomes the deciding factor, but only after it's
been through this same notebook and held up. Opus is more than a 1–3 sentence summary
needs. The numbers below show cost isn't really the constraint anyway.

In [23]:
# ~250-token system prompt + ~550-token context in, ~90 tokens out (from Task 7).
IN_TOK, OUT_TOK = 800, 90
price = {"claude-haiku-4-5": (1, 5),
         "claude-sonnet-4-6": (3, 15),
         "claude-opus-4-8": (5, 25)}

def cost(model, n, batch=True):
    pin, pout = price[model]
    c = n*IN_TOK/1e6*pin + n*OUT_TOK/1e6*pout
    return c * (0.5 if batch else 1)

for n, label in [(12215, "12,215 high-risk"), (40771, "40,771 (all)")]:
    print(f"{label} backfill, batched:")
    for m in price:
        print(f"  {m:18} ${cost(m, n):6.2f}")
    print()
print(f"Sonnet for the high-risk backfill: ${cost('claude-sonnet-4-6', 12215):.2f} "
      f"(${cost('claude-sonnet-4-6', 12215, batch=False):.2f} unbatched)")

12,215 high-risk backfill, batched:
  claude-haiku-4-5   $  7.63
  claude-sonnet-4-6  $ 22.90
  claude-opus-4-8    $ 38.17

40,771 (all) backfill, batched:
  claude-haiku-4-5   $ 25.48
  claude-sonnet-4-6  $ 76.45
  claude-opus-4-8    $127.41

Sonnet for the high-risk backfill: $22.90 ($45.81 unbatched)


### Beyond the batch job

Right now this is an overnight batch, so latency doesn't matter and the Batches API
(half price, usually done within the hour) is the obvious fit. If this ever becomes a
chat interface, the requirement flips to interactive latency — I'd stream the answer
and cache the shared prompt + elevator context so follow-up questions about the same
elevator are nearly free.

Either way I'd keep this notebook as the gate: any model or prompt change re-runs the
grader against the database and has to come back with zero majors and zero
hallucinations before it ships. And explanations only get regenerated when an
elevator's data changes — the column is already the cache.

### Running it on another model

It's one call. Generate the candidate's explanations into a column (or a dict) and:

In [24]:
def evaluate_model(explanations, t=truth, label="candidate"):
    d = defaultdict(int)
    for e in explanations:
        if e in t:
            s, _ = grade(e, explanations[e], t[e]); d[s] += 1
    return {
        "model": label,
        "accuracy": {s.name: d[s] for s in Score},
        "ship_ok": d[Score.MAJOR] == 0 and d[Score.HALLUCINATION] == 0,
    }

# current production output, for reference
print(json.dumps(evaluate_model(expl, label=MODEL_LABEL), indent=2))
conn.close()

{
  "model": "llama3.1:8b-instruct-q8_0",
  "accuracy": {
    "CORRECT": 9,
    "MINOR": 0,
    "MAJOR": 1,
    "HALLUCINATION": 0
  },
  "ship_ok": false
}
